# Validation: does the model return sensible comparables?

Every number in the README's validation section comes from this notebook. The
checks run cheapest-first: if feature resolution or the name join is broken,
nothing after it means anything.

Run the pipeline once before opening it:

```bash
python run_pipeline.py --stage all
```

In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src import config, paths, similarity
from src.name_matching import normalise_name

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

features = pd.read_parquet(paths.FEATURES)
percentiles = pd.read_parquet(paths.PERCENTILES)
comps = pd.read_parquet(paths.SIMILARITY)
values = pd.read_parquet(paths.VALUE_PREDICTIONS)
models = joblib.load(paths.SIMILARITY_MODELS)
seasons = dict(zip(features["player_season_id"], features["season"].astype(str)))

# Accent-insensitive name search: "Luis Diaz" must find FBref's "Luis Díaz".
features["name_key"] = features["player"].map(normalise_name)


def find(name, season=None):
    """Rows for a player, ignoring accents and case.

    An exact name match wins; only if there is none does it fall back to a
    substring match, so "Rodri" finds Rodri rather than Rodrigo Ely.
    """
    key = normalise_name(name)
    rows = features[features["name_key"] == key]
    if rows.empty:
        rows = features[features["name_key"].str.contains(key, regex=False)]
    if season is not None:
        rows = rows[rows["season"].astype(str) == str(season)]
    return rows


print(f"{len(features):,} player-seasons in the model")
print(features.pivot_table(index="league", columns="season", values="player", aggfunc="size", fill_value=0))
print()
print(features.groupby("position_group").size().sort_values().to_string())

3,768 player-seasons in the model
season              2223  2324  2425
league                              
ENG-Premier League   358   311   340
ESP-La Liga          370     0   372
FRA-Ligue 1          355     0   316
GER-Bundesliga       300     0   304
ITA-Serie A          375     0   367

position_group
AM    263
DM    345
W     499
ST    535
CM    599
FB    676
CB    851


## 1. Did every feature resolve?

FBref renames table headers from time to time. A feature that fails to resolve
is dropped from every group that uses it, which weakens the profiles without
raising an error, so the resolution report is checked first.

In [2]:
resolution = pd.read_csv(paths.FEATURE_RESOLUTION)
missing = resolution[resolution["status"] == "MISSING"]

print(f"{len(resolution) - len(missing)}/{len(resolution)} features resolved")
if len(missing):
    print("\nUnresolved (fix the patterns in src/config.py against reports/fbref_columns.txt):")
    display(missing[["feature", "patterns_tried"]])

77/77 features resolved


## 2. What did the name join achieve?

FBref and Transfermarkt are joined on name and birth year, then every match is
confirmed against the club the player actually appeared for that season. A low
match rate would silently remove exactly the recognisable players a case study
needs, so the rate is reported rather than assumed.

In [3]:
summary = json.loads(paths.MATCH_SUMMARY.read_text())
print(f"Name match rate: {summary['match_rate']:.1%} of {summary['total']:,} distinct players")
print(summary["decisions"])

player_seasons = pd.read_parquet(paths.PLAYER_SEASONS, columns=["club_check"])
print("\nClub check:", player_seasons["club_check"].value_counts().to_dict())

print(f"\nModel rows with a market value: {features['market_value_eur'].notna().mean():.1%}")
print(f"Model rows with a Transfermarkt position: {(features['position_source'] == 'transfermarkt').mean():.1%}")
print(f"Model rows with a dated contract: {features['contract_years_remaining'].notna().mean():.1%}")

Name match rate: 96.0% of 3,779 distinct players
{'accepted': 3629, 'no_candidates': 69, 'review': 41, 'rejected': 40}

Club check: {'confirmed': 5620, 'recovered': 80, 'corrected': 52, 'unmatched': 28, 'unlinked': 6, 'unconfirmed': 3}

Model rows with a market value: 99.5%
Model rows with a Transfermarkt position: 99.7%
Model rows with a dated contract: 87.9%


In [4]:
# Spot-check that well-known players survived the join with a value attached.
WATCHLIST = ["Alexis Mac Allister", "Bruno Guimaraes", "Milos Kerkez", "Nico Williams", "Joshua Kimmich"]

for name in WATCHLIST:
    rows = find(name)
    if rows.empty:
        print(f"MISSING  {name}")
        continue
    row = rows.nlargest(1, "minutes").iloc[0]
    flag = "" if pd.notna(row["market_value_eur"]) else "   <-- no market value joined"
    print(f"ok       {name:24s} {row['position_group']:3s} {row['team']}{flag}")

ok       Alexis Mac Allister      CM  Brighton
ok       Bruno Guimaraes          CM  Newcastle
ok       Milos Kerkez             FB  Bournemouth
ok       Nico Williams            W   Athletic Club
ok       Joshua Kimmich           DM  Bayern Munich


## 3. Style dimensions

The top and bottom loadings of the first three components per group. These are
what the README's style-dimension table names in football terms.

In [5]:
loadings = pd.read_csv(paths.PCA_LOADINGS)


def describe(group, n=4):
    chunk = loadings[loadings["position_group"] == group]
    for component in chunk["component"].unique()[:3]:
        block = chunk[chunk["component"] == component].sort_values("loading", ascending=False)
        variance = block["explained_variance_ratio"].iloc[0]
        print(f"\n{group} {component}  ({variance:.0%} of variance)")
        print("  high:", ", ".join(f"{r.feature} {r.loading:+.2f}" for r in block.head(n).itertuples()))
        print("  low: ", ", ".join(f"{r.feature} {r.loading:+.2f}" for r in block.tail(n).itertuples()))


for group in config.POSITION_GROUPS:
    describe(group)


CB PC1  (20% of variance)
  high: passes_attempted +0.44, prog_passes +0.43, prog_carries +0.35, tackles_won +0.35
  low:  aerial_win_pct +0.01, shots_blocked -0.02, aerials_won -0.09, clearances -0.13

CB PC2  (15% of variance)
  high: shots_blocked +0.45, clearances +0.42, pass_completion_pct +0.35, aerial_win_pct +0.29
  low:  tackles_def_third -0.17, tackles_won -0.18, ball_recoveries -0.19, prog_carries -0.19

CB PC3  (12% of variance)
  high: tackles_def_third +0.40, interceptions +0.37, aerials_won +0.35, tackles_won +0.35
  low:  prog_carries -0.13, passes_attempted -0.17, pass_completion_long_pct -0.27, pass_completion_pct -0.32

FB PC1  (36% of variance)
  high: touches_att_third +0.36, passes_into_box +0.34, xa +0.32, prog_passes_received +0.32
  low:  tackles_plus_interceptions +0.05, take_on_success_pct +0.03, pass_completion_pct +0.00, dribblers_tackled_pct -0.10

FB PC2  (18% of variance)
  high: prog_carry_distance +0.38, pass_completion_pct +0.37, carries_into_final_t

## 4. The eye test

Top five comparables for players whose game is well known. What failure looks
like: a target man returning false nines (the features measure output, not
role), or a deep-lying playmaker returning every high-volume passer (they lack
the zone and progression detail that separates a regista from a metronome).

In [6]:
def comps_for(name, season=None, n=5):
    pool = find(name, season)
    if pool.empty:
        print(f"Not found: {name}")
        return None
    row = pool.nlargest(1, "minutes").iloc[0]
    table = similarity.top_comps(row["player_season_id"], comps, n=n)
    out = table.merge(
        features[["player_season_id", "player", "team", "season", "age", "market_value_eur"]]
        .rename(columns={"player_season_id": "comp_player_season_id"}),
        on="comp_player_season_id",
    )
    print(f"\n=== {row['player']} ({row['team']}, {row['season']}, {row['position_group']}) ===")
    return out[["rank", "player", "team", "season", "age", "similarity", "market_value_eur"]]


EYE_TEST = [
    ("Rodri", "2223"),             # DM - should return controllers, not destroyers
    ("Virgil van Dijk", "2425"),   # CB - should return ball-playing centre-backs
    ("Mohamed Salah", "2425"),     # W  - should return high-volume shooters
    ("Erling Haaland", "2425"),    # ST - should NOT return false nines
    ("Alexander-Arnold", "2425"),  # FB - should return creative, inverted full-backs
]

for name, season in EYE_TEST:
    table = comps_for(name, season)
    if table is not None:
        display(table)


=== Rodri (Manchester City, 2223, DM) ===


,rank,player,team,season,age,similarity,market_value_eur
0,1,Thomas Partey,Arsenal,2223,29.552361,0.945329,38000000.0
1,3,Nabil Bentaleb,Angers,2223,28.104038,0.865736,5000000.0
2,4,Sergio Busquets,Barcelona,2223,34.461328,0.847212,5000000.0
3,5,Baptiste Santamaria,Rennes,2223,27.816564,0.815858,16000000.0
4,6,Pierre Højbjerg,Marseille,2425,29.409993,0.808432,20000000.0



=== Virgil van Dijk (Liverpool, 2425, CB) ===


,rank,player,team,season,age,similarity,market_value_eur
0,1,Jonathan Tah,Leverkusen,2425,28.889802,0.887051,30000000.0
1,3,Alexsandro Ribeiro,Lille,2425,25.399042,0.819817,12000000.0
2,4,Amir Rrahmani,Napoli,2425,30.852841,0.776177,12000000.0
3,5,Stefan de Vrij,Inter,2425,32.906229,0.770502,7000000.0
4,6,Dylan Batubinsika,Saint-Étienne,2425,28.878850,0.767496,2500000.0



=== Mohamed Salah (Liverpool, 2425, W) ===


,rank,player,team,season,age,similarity,market_value_eur
0,2,Sadio Mané,Bayern Munich,2223,30.726899,0.940554,60000000.0
1,3,Gabriel Martinelli,Arsenal,2223,21.538672,0.937513,60000000.0
2,5,Mason Greenwood,Marseille,2425,23.252567,0.917636,35000000.0
3,6,Ademola Lookman,Atalanta,2425,27.200548,0.911177,55000000.0
4,7,Bradley Barcola,Paris SG,2425,22.332649,0.906542,65000000.0



=== Erling Haaland (Manchester City, 2425, ST) ===


,rank,player,team,season,age,similarity,market_value_eur
0,3,Youssoufa Moukoko,Dortmund,2223,18.113621,0.930579,30000000.0
1,4,Alexander Sørloth,Atlético Madrid,2425,29.075975,0.922961,25000000.0
2,5,Ollie Watkins,Aston Villa,2425,29.007529,0.918358,55000000.0
3,6,Robert Lewandowski,Barcelona,2425,36.364134,0.897320,15000000.0
4,7,Ollie Watkins,Aston Villa,2324,28.005476,0.875091,60000000.0



=== Trent Alexander-Arnold (Liverpool, 2425, FB) ===


,rank,player,team,season,age,similarity,market_value_eur
0,1,Kieran Trippier,Newcastle Utd,2223,32.284736,0.964657,13000000.0
1,2,Przemysław Frankowski,Lens,2223,27.723477,0.958903,7000000.0
2,4,Jordi Alba,Barcelona,2223,33.782341,0.946123,5000000.0
3,5,Cristiano Biraghi,Fiorentina,2223,30.332649,0.945414,6500000.0
4,6,Pedro Porro,Tottenham,2425,25.303217,0.940794,45000000.0


**Verdicts** (as in the README):

- Rodri - Partey, Bentaleb, Busquets, Santamaria, Højbjerg: deep controllers that
  like to control the game. Hit.
- Virgil van Dijk - Tah, Alexsandro, Rrahmani, de Vrij: composed centre-backs from
  possession sides. Hit.
- Mohamed Salah - Mané, Martinelli, Greenwood, Lookman, Barcola: wingers who look
  to score first and pass second, also known as inside forwards. Hit.
- Erling Haaland - Moukoko, Sørloth, Watkins, Lewandowski: classic strikers, great
  first touch, dominate physically and good finishers. Hit.
- Trent Alexander-Arnold - Trippier, Frankowski, Alba, Biraghi, Porro: **miss.**
  The full-back stats see crossing volume and chance creation, but nothing
  records *where* a pass is played from, so a full-back who inverts to play long
  switches looks like a byline crosser.

## 5. Self-consistency

The same player a season later is the best comparable there is. For every
player with two seasons in the same group, `similarity.own_season_ranks` ranks
everyone in the group against him and records where his own other season
lands - computed exactly from the PCA scores, so a season ranked outside the
stored neighbour list counts as a miss instead of disappearing. A random
embedding would place it around the middle of the group (50%).

In [7]:
ranks = similarity.own_season_ranks(models, seasons)
ranks["rank_pct"] = ranks["rank"] / ranks["group_size"]

print(json.dumps(json.loads(paths.VALIDATION.read_text()), indent=2))

summary_by_gap = ranks.groupby("season_gap").agg(
    scored=("rank", "size"),
    median_rank=("rank", "median"),
    median_rank_pct=("rank_pct", "median"),
    top_5pct_share=("rank_pct", lambda s: (s <= 0.05).mean()),
)
display(summary_by_gap.round(3))

ax = ranks.pivot(columns="season_gap", values="rank_pct").plot(
    kind="hist", bins=40, alpha=0.6, figsize=(7, 3),
    title="Own other season, as a share of the group (lower is better)",
)
ax.set_xlabel("rank / group size")

{
  "median_rank": 49.0,
  "top_5_share": 0.16585581601061478,
  "median_rank_pct": 0.09048178613396005,
  "top_5pct_share": 0.39451570101724903,
  "player_seasons_scored": 2261,
  "by_season_gap": {
    "1": {
      "median_rank": 19.0,
      "top_5_share": 0.28987993138936535,
      "median_rank_pct": 0.0333889816360601,
      "top_5pct_share": 0.5883361921097771,
      "player_seasons_scored": 583
    },
    "2": {
      "median_rank": 68.5,
      "top_5_share": 0.12276519666269368,
      "median_rank_pct": 0.11834319526627218,
      "top_5pct_share": 0.32717520858164484,
      "player_seasons_scored": 1678
    }
  }
}


,scored,median_rank,median_rank_pct,top_5pct_share
season_gap,,,,
1,583,19.0,0.033,0.588
2,1678,68.5,0.118,0.327


## 6. Minutes sensitivity

The model includes everyone above 600 minutes and the app defaults its slider
to 900. How much do the top-5 comparables change between the two pools? High
churn would mean low-minutes players are adding noise.

In [8]:
TIGHT = config.MIN_MINUTES                # 900
LOOSE = config.MIN_MINUTES_SENSITIVITY    # 600

tight_ids = set(features.loc[features["minutes"] >= TIGHT, "player_season_id"])
others = comps[~comps["same_player"]]

overlaps = []
for player_season_id in sorted(tight_ids)[:400]:
    mine = others[others["player_season_id"] == player_season_id]
    loose_top5 = set(mine.nsmallest(5, "rank")["comp_player_season_id"])
    tight_top5 = set(mine[mine["comp_player_season_id"].isin(tight_ids)].nsmallest(5, "rank")["comp_player_season_id"])
    overlaps.append(len(loose_top5 & tight_top5))

print(f"Mean overlap of top-5 comparables, {LOOSE}+ pool vs {TIGHT}+ pool: {np.mean(overlaps):.2f} / 5")

Mean overlap of top-5 comparables, 600+ pool vs 900+ pool: 4.38 / 5


## 7. Is the value model learning more than league and age?

The characteristic failure: an undervalued shortlist that is all 33-year-olds
in the cheapest league, because the model learnt league and age and nothing
else. Compare the flagged players with the pool.

In [9]:
metrics = json.loads(paths.VALUE_METRICS.read_text())
display(pd.DataFrame({k: v for k, v in metrics.items() if k != "selected_model"}).round(3))
print("selected:", metrics["selected_model"])

flagged = values[values["undervalued"]]
display(pd.DataFrame({
    "pool": values["league"].value_counts(normalize=True),
    "flagged": flagged["league"].value_counts(normalize=True),
}).round(3))
print("Mean log residual by league (close to 0 = no league bias):")
print(values.groupby("league")["log_residual"].mean().round(3).to_string())

,ridge,gbm
cv_mae_log,0.596,0.601
cv_r2_log,0.615,0.606
median_absolute_percentage_error,45.022,45.459
median_ratio,0.968,0.967
median_absolute_error_eur,3373886.199,3403347.371


selected: ridge


,pool,flagged
league,,
ENG-Premier League,0.268,0.202
ESP-La Liga,0.196,0.206
FRA-Ligue 1,0.178,0.201
GER-Bundesliga,0.161,0.175
ITA-Serie A,0.197,0.215


Mean log residual by league (close to 0 = no league bias):
league
ENG-Premier League   -0.000
ESP-La Liga           0.000
FRA-Ligue 1          -0.006
GER-Bundesliga        0.000
ITA-Serie A           0.002


### Backtest: did the 2022-23 flags rise in value?

The test of the value flag that does not rely on the model's own numbers. For
every 2022-23 player-season, compare the market value used in the model with
the player's last Transfermarkt valuation on or before 1 July 2025, two
seasons later. Split by age, because young players rise anyway.

Two caveats: only players still valued in 2025 are counted, and part of what
the flag catches is Transfermarkt lagging behind breakthrough seasons - which
is itself useful to a recruiter, but is not the same as the model knowing
something the market cannot.

In [10]:
valuations = pd.read_parquet(paths.TM_VALUATIONS).sort_values("date")
later = valuations[valuations["date"] <= "2025-07-01"].groupby("player_id").last()

cohort = values[(values["season"].astype(str) == "2223") & values["tm_player_id"].notna()].copy()
cohort["value_2025"] = cohort["tm_player_id"].map(later["market_value_in_eur"])
cohort["valued_on"] = cohort["tm_player_id"].map(later["date"])
cohort = cohort[cohort["valued_on"] >= "2025-01-01"]
cohort["growth"] = cohort["value_2025"] / cohort["market_value_eur"]
cohort["age_band"] = pd.cut(cohort["age"], [0, 22, 26, 29, 50], labels=["<=22", "23-26", "27-29", "30+"])

backtest = cohort.groupby(["age_band", "undervalued"], observed=True)["growth"].agg(
    players="size", median_growth="median", share_rose=lambda g: (g > 1).mean(),
)
display(backtest.round(2))
display(cohort.groupby("undervalued")["growth"].agg(
    players="size", median_growth="median", share_rose=lambda g: (g > 1).mean()).round(2))

players  median_growth  share_rose
age_band undervalued                                    
<=22     False            146           1.40        0.68
         True              48           4.75        0.92
23-26    False            423           0.84        0.39
         True             134           1.45        0.65
27-29    False            289           0.48        0.10
         True              76           0.75        0.36
30+      False            274           0.33        0.02
         True              69           0.50        0.09

,players,median_growth,share_rose
undervalued,,,
False,1132,0.60,0.26
True,327,1.06,0.50


## 8. Known transfers

Real replacement signings: would the model have surfaced the replacement as a
comparable to the player he replaced? The replacement's season *before* the
move is the fair test; his season at the new club is shown too, because ranking
high there means he went on to play the departed player's role.

In [11]:
# (departed, his last season before leaving, replacement, replacement's season before joining)
TRANSFER_PAIRS = [
    ("Granit Xhaka", "2223", "Declan Rice", "2223"),       # Arsenal 2023
    ("Gundogan", "2223", "Kovacic", "2223"),               # Manchester City 2023
    ("Fabinho", "2223", "Wataru Endo", "2223"),            # Liverpool 2023
    ("Karim Benzema", "2223", "Joselu", "2223"),           # Real Madrid 2023
    ("Alexander-Arnold", "2425", "Frimpong", "2425"),      # Liverpool 2025
    ("Thomas Partey", "2425", "Zubimendi", "2425"),        # Arsenal 2025
    ("De Bruyne", "2425", "Cherki", "2425"),               # Manchester City 2025
    ("De Bruyne", "2425", "Reijnders", "2425"),            # Manchester City 2025
]


def rank_in_group(departed_id, candidate_ids):
    """Exact rank (1 = closest) of each candidate among the departed player's group."""
    group = features.loc[features["player_season_id"] == departed_id, "position_group"].iloc[0]
    model = models[group]
    ids = list(model["ids"])
    unit = model["scores"] / np.linalg.norm(model["scores"], axis=1, keepdims=True)
    sims = unit @ unit[ids.index(departed_id)]
    sims[ids.index(departed_id)] = -np.inf
    out = {}
    for cid in candidate_ids:
        out[cid] = int((sims > sims[ids.index(cid)]).sum()) + 1 if cid in ids else None
    return group, len(ids), out


rows = []
for departed, d_season, replacement, r_season in TRANSFER_PAIRS:
    d = find(departed, d_season)
    if d.empty:
        rows.append({"departed": departed, "replacement": replacement, "result": "departed not in sample"})
        continue
    d = d.nlargest(1, "minutes").iloc[0]
    r = find(replacement)
    group, size, ranked = rank_in_group(d["player_season_id"], list(r["player_season_id"]))
    before = r[r["season"].astype(str) == r_season]
    before_rank = ranked.get(before["player_season_id"].iloc[0]) if len(before) else None
    best_id = min((k for k, v in ranked.items() if v), key=ranked.get, default=None)
    rows.append({
        "departed": f"{d['player']} ({d['season']}, {group})",
        "replacement": replacement,
        "replacement groups": ", ".join(sorted(set(r["position_group"]))),
        "rank before move": before_rank,
        "best rank, any season": f"{ranked[best_id]} ({seasons[best_id]})" if best_id else None,
        "group size": size,
    })

display(pd.DataFrame(rows))

,departed,replacement,replacement groups,rank before move,"best rank, any season",group size
0,"Granit Xhaka (2223, DM)",Declan Rice,CM,NaN,NaN,345
1,"İlkay Gündoğan (2223, CM)",Kovacic,CM,48.0,48 (2223),599
2,"Fabinho (2223, DM)",Wataru Endo,DM,254.0,1 (2324),345
3,"Karim Benzema (2223, ST)",Joselu,ST,325.0,325 (2223),535
4,"Trent Alexander-Arnold (2425, FB)",Frimpong,FB,167.0,167 (2425),676
5,"Thomas Partey (2425, DM)",Zubimendi,DM,59.0,59 (2425),345
6,"Kevin De Bruyne (2425, AM)",Cherki,AM,12.0,12 (2425),263
7,"Kevin De Bruyne (2425, AM)",Reijnders,CM,NaN,NaN,263


## 9. Playing roles

Every player-season is scored against its group's roles (`config.ROLES`); the
best fit is his label, and a runner-up within
`config.ROLE_HYBRID_MARGIN` points makes him a hybrid. Two checks:

* **Stability.** A role describes how a player plays, so it should mostly
  survive into his next season. "Chance" is the agreement expected if roles
  were dealt at random in the observed proportions.
* **Eye test.** The top regulars per role in 2024-25 should read like the
  role's name.

In [12]:
roles = pd.read_parquet(paths.ROLES)
role_scores = pd.read_parquet(paths.ROLE_SCORES)
stability = pd.read_csv(paths.ROLE_STABILITY)
display(stability)
print(f"Hybrids: {roles['hybrid'].mean():.0%} of player-seasons")
print(roles.groupby("position_group")["primary_role"].value_counts(normalize=True).round(2).to_string())

,position_group,pairs,roles,same_primary,same_any,chance_primary
0,CB,298,4.0,0.520,0.644,0.255
1,FB,244,3.0,0.598,0.701,0.352
2,DM,120,4.0,0.608,0.692,0.259
3,CM,204,4.0,0.603,0.730,0.260
4,AM,93,3.0,0.677,0.763,0.346
5,W,165,4.0,0.606,0.739,0.272
6,ST,157,4.0,0.580,0.732,0.254
7,All,1281,NaN,0.586,0.705,0.283


Hybrids: 18% of player-seasons
position_group  primary_role         
AM              Advanced playmaker       0.39
                Second striker           0.35
                Channel midfielder       0.26
CB              No-nonsense CB           0.33
                CB                       0.23
                Ball-playing CB          0.23
                Wide CB                  0.20
CM              Deep-lying playmaker     0.31
                Box-to-box               0.26
                Advanced playmaker       0.22
                Midfield playmaker       0.21
DM              Half-back                0.32
                Deep-lying playmaker     0.26
                Ball-winner              0.21
                Anchor                   0.21
FB              Overlapping wing-back    0.41
                Full-back                0.36
                Inverted full-back       0.23
ST              Target man               0.30
                False 9                  0.28
           

In [13]:
top = pd.read_csv(paths.ROLE_TOP_PLAYERS)
top = top[top["season"].astype(str) == "2425"]
display(top.groupby(["position_group", "role"], sort=False)["player"]
        .agg(lambda names: "; ".join(names)).rename("top five, 2024-25").to_frame())

top five, 2024-25
position_group role                                                                                                          
AM             Second striker                              Jamal Musiala; Dani Olmo; Rémy Cabella; Oihan Sancet; Fermin López
               Advanced playmaker                 Rayan Cherki; Giovani Lo Celso; Kevin De Bruyne; Florian Wirtz; Lee Kang-in
               Channel midfielder                         Léo Scienza; Jamal Musiala; Florian Wirtz; Sávio; Gaetano Oristanio
CB             Ball-playing CB                         Manuel Akanji; Dayot Upamecano; Pierre Kalulu; Rúben Dias; Daley Blind
               No-nonsense CB                          Jubal; James Tarkowski; Andreas Hanche-Olsen; Renato Veiga; Pablo Marí
               CB                                             Clinton Mata; José Luis Palomino; Tuta; Enzo Boyomo; Yerry Mina
               Wide CB                        Daley Blind; Joško Gvardiol; Ramy Bensebaini; Marcos Alonso; Alessandro Bastoni
CM             Box-to-box                        Fabián Ruiz Peña; João Neves; Adrien Thomasson; Leon Goretzka; Adrien Rabiot
               Midfield playmaker                        Diego Moreira; Andy Diouf; Khéphren Thuram; Yunus Musah; Aron Dønnum
               Deep-lying playmaker                Dani Ceballos; Leon Goretzka; Aleix García; Frenkie de Jong; Daniel Parejo
               Advanced playmaker                      Rodrigo De Paul; Arne Maier; Luis Henrique; Nadiem Amiri; Ludovic Blas
DM             Ball-winner                     Soungoutou Magassa; Manuel Ugarte; Benjamin André; Tyler Adams; Moisés Caicedo
               Deep-lying playmaker                  Joshua Kimmich; Manuel Locatelli; Granit Xhaka; Leandro Paredes; Vitinha
               Half-back               Aurélien Tchouaméni; Nemanja Gudelj; Stanko Jurić; Omar Mascarell; Edimilson Fernandes
               Anchor                                                 Kaishū Sano; Marc Casado; Carlos Baleba; André; Vitinha
FB             Overlapping wing-back        Álex Grimaldo; Nuno Tavares; Caio Henrique; Maximilian Mittelstädt; Achraf Hakimi
               Inverted full-back     Alphonso Davies; Kyle Walker; Konrad Laimer; Lutsharel Geertruida; Michael Amir Murillo
               Full-back                                  Wilfried Singo; Noussair Mazraoui; Guéla Doué; Adam Obert; Bernardo
ST             Poacher                             Cristhian Stuani; Emanuel Emegha; Gonçalo Ramos; Mika Biereth; Myron Boadu
               Target man                               Jonas Wind; Philipp Hofmann; Juanmi Latasa; Vedat Muriqi; Milan Đurić
               False 9                      Isaac Palazón Camacho; Iago Aspas; Paulo Dybala; Matheus Cunha; Giacomo Raspadori
               Channel runner                            Moise Kean; Kylian Mbappé; Hugo Ekitike; Ayoze Pérez; Alexander Isak
W              Traditional winger                      Junya Itō; Bryan Zaragoza; Dwight McNeil; Iván Sánchez; Antoine Joujou
               Inside forward                          Luis Díaz; Mohamed Salah; Vinicius Júnior; Noni Madueke; Karim Adeyemi
               Wide playmaker                              Thiago Almada; Vincenzo Grifo; Álex Baena; Michael Olise; Raphinha
               1v1 dribbler                            Jeremy Doku; Lamine Yamal; Chidera Ejuke; Jamie Gittens; Takefusa Kubo

In [14]:
def role_history(name):
    """A player's role label in each of his seasons."""
    rows = find(name).merge(roles, on=["player_season_id", "position_group"], how="left",
                            suffixes=("", "_role"))
    return rows[["player", "team", "season", "position_group", "role_label",
                 "primary_score", "secondary_score"]].sort_values("season")


for name in ["Declan Rice", "Joshua Kimmich", "Bukayo Saka", "Mohamed Salah", "Serhou Guirassy"]:
    display(role_history(name))

,player,team,season,position_group,role_label,primary_score,secondary_score
0,Declan Rice,West Ham,2223,CM,Deep-lying playmaker,96.785714,82.857143
1,Declan Rice,Arsenal,2324,CM,Deep-lying playmaker,82.222222,75.555556
2,Declan Rice,Arsenal,2425,CM,Box-to-box / Advanced playmaker,93.065693,88.686131


,player,team,season,position_group,role_label,primary_score,secondary_score
0,Joshua Kimmich,Bayern Munich,2223,DM,Deep-lying playmaker / Ball-winner,93.125,90.000000
1,Joshua Kimmich,Bayern Munich,2425,DM,Deep-lying playmaker,100.000,85.620915


,player,team,season,position_group,role_label,primary_score,secondary_score
0,Bukayo Saka,Arsenal,2223,W,Inside forward / 1v1 dribbler,77.375566,72.398190
1,Bukayo Saka,Arsenal,2324,W,Wide playmaker,82.926829,63.414634
2,Bukayo Saka,Arsenal,2425,W,1v1 dribbler / Inside forward,96.202532,93.670886


,player,team,season,position_group,role_label,primary_score,secondary_score
0,Mohamed Salah,Liverpool,2223,W,Inside forward,97.285068,76.470588
1,Mohamed Salah,Liverpool,2324,W,Wide playmaker / Inside forward,87.804878,85.365854
2,Mohamed Salah,Liverpool,2425,W,Inside forward,99.578059,81.012658


,player,team,season,position_group,role_label,primary_score,secondary_score
0,Serhou Guirassy,Stuttgart,2223,ST,Target man,90.038314,66.666667
1,Serhou Guirassy,Dortmund,2425,ST,Target man,93.617021,81.702128


---

## Numbers used in the README

In [15]:
validation = json.loads(paths.VALIDATION.read_text())
gap1 = validation["by_season_gap"]["1"]
ridge = metrics[metrics["selected_model"]]
print(f"Player-seasons in the model:          {len(features):,}")
print(f"Name match rate:                      {summary['match_rate']:.1%}")
print(f"Own adjacent season, median rank:     {gap1['median_rank']:.0f} (top {gap1['median_rank_pct']:.1%} of the group)")
print(f"Own adjacent season in top 5% of group: {gap1['top_5pct_share']:.1%}")
print(f"Value model median abs. error (CV):   {ridge['median_absolute_percentage_error']:.1f}%")
print(f"Smallest position group:              {features['position_group'].value_counts().idxmin()} "
      f"({features['position_group'].value_counts().min()} player-seasons)")
overall = stability.set_index("position_group").loc["All"]
print(f"Same role next season:                {overall['same_primary']:.0%} "
      f"({overall['same_any']:.0%} counting hybrids) vs {overall['chance_primary']:.0%} by chance")
print(f"Hybrid player-seasons:                {roles['hybrid'].mean():.0%}")

Player-seasons in the model:          3,768
Name match rate:                      96.0%
Own adjacent season, median rank:     19 (top 3.3% of the group)
Own adjacent season in top 5% of group: 58.8%
Value model median abs. error (CV):   45.0%
Smallest position group:              AM (263 player-seasons)
Same role next season:                59% (70% counting hybrids) vs 28% by chance
Hybrid player-seasons:                18%
